# TF-IDF + Logistic Regression

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bethesda-classification")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

## Служебная информация об эксперименте

Фиксируем имя участника, название задачи, дату и текущий Git commit.

Git commit используется как идентификатор версии кода. Перед запуском модели
необходимо убедиться, что код эксперимента закоммичен. Если в рабочей директории
есть незакоммиченные изменения, текущий commit не полностью описывает состояние
кода, на котором выполняется эксперимент.

In [10]:
import subprocess
from datetime import datetime


participant = "Savely"
task_name = "bethesda-classification"

# Получаем SHA текущего Git-коммита
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

# Проверяем наличие незакоммиченных изменений
git_status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    text=True,
).strip()

run_date = datetime.now().strftime("%d-%m-%Y")

print("Participant:", participant)
print("Task:", task_name)
print("Date:", run_date)
print("Git commit:", git_commit)

if git_status:
    print("\nWARNING: в репозитории есть незакоммиченные изменения.")
    print("Перед первым MLflow run необходимо сделать commit и повторно выполнить эту ячейку.")
else:
    print("\nGit working tree clean.")

Participant: Savely
Task: bethesda-classification
Date: 11-10-2026
Git commit: 584ffa646e95c41d70095969957641d897fd5e39

Перед первым MLflow run необходимо сделать commit и повторно выполнить эту ячейку.


## Версия датасета

Для экспериментов используется актуальный `prepared_dataset.csv`.

Версия данных определяется полным SHA256-хешем файла. Если содержимое датасета
изменится хотя бы на уровне одного байта, изменится и его SHA256.

Полный SHA256 будет сохраняться в каждом MLflow run как `dataset_version`.
Первые 12 символов используются только для удобного отображения и имени
архивной копии датасета.

In [3]:
from pathlib import Path

from src.common.dataset_versioning import calculate_sha256


# Определяем корень проекта
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = next(
        path
        for path in PROJECT_ROOT.parents
        if (path / "src").exists()
    )

# Актуальный воспроизводимый датасет
dataset_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "prepared_dataset.csv"
)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Датасет не найден: {dataset_path}"
    )

# Вычисляем версию данных
dataset_hash = calculate_sha256(dataset_path)
dataset_version_short = dataset_hash[:12]

# Ожидаемая архивная копия этой версии
dataset_version_path = (
    dataset_path.parent
    / "versions"
    / f"{dataset_path.stem}_{dataset_version_short}{dataset_path.suffix}"
)

print("Dataset:", dataset_path.name)
print("Dataset version:", dataset_version_short)
print("Full SHA256:", dataset_hash)
print("Archived version:", dataset_version_path.name)
print("Archive exists:", dataset_version_path.exists())

Dataset: prepared_dataset.csv
Dataset version: e88b34b729ea
Full SHA256: e88b34b729eaf057b6f2a7deeae7fcfce901da710ddce628a329d9d96aaad724
Archived version: prepared_dataset_e88b34b729ea.csv
Archive exists: True


## Загрузка и первичная проверка датасета

Загрузим актуальный `prepared_dataset.csv`, для которого выше был вычислен
SHA256-хеш.

Проверим размер датасета, структуру столбцов и первые строки. На этом этапе
модель ещё не обучается, поэтому отдельный MLflow run не создаётся.

In [8]:
import pandas as pd

df = pd.read_csv(dataset_path, sep=";")

print("Путь к датасету:", dataset_path)
print("Размер датасета:", df.shape)
print("Столбцы:", df.columns.tolist())

Путь к датасету: /Users/savely/Desktop/diploma/data/processed/prepared_dataset.csv
Размер датасета: (5613, 8)
Столбцы: ['case_id', 'source_table', 'source_row', 'description', 'conclusion', 'bethesda', 'duplicate_group', 'duplicate_type']


## Проверка структуры и качества данных

Перед построением модели проверим:

- наличие необходимых столбцов;
- количество пропущенных значений;
- число объектов каждого класса Bethesda;
- долю каждого класса в датасете.

Это позволит оценить состояние нового датасета и степень дисбаланса классов
перед разделением на train и test.

In [5]:
print("Столбцы:")
print(df.columns.tolist())

print("\nПропущенные значения:")
print(df.isna().sum())

print("\nКоличество объектов каждого класса Bethesda:")
print(df["bethesda"].value_counts().sort_index())

print("\nДоля каждого класса:")
print(
    df["bethesda"]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Столбцы:
['case_id', 'source_table', 'source_row', 'description', 'conclusion', 'bethesda', 'duplicate_group', 'duplicate_type']

Пропущенные значения:
case_id               0
source_table          0
source_row            0
description           0
conclusion            0
bethesda              0
duplicate_group    5603
duplicate_type     5603
dtype: int64

Количество объектов каждого класса Bethesda:
bethesda
I       575
II     3825
III     286
IV      559
V       172
VI      196
Name: count, dtype: int64

Доля каждого класса:
bethesda
I      0.1024
II     0.6815
III    0.0510
IV     0.0996
V      0.0306
VI     0.0349
Name: proportion, dtype: float64


## Проверка дубликатов

В актуальном датасете информация о найденных дубликатах сохраняется в полях
`duplicate_group` и `duplicate_type`.

Дубликаты не удаляются автоматически на этапе подготовки данных. Перед обучением
необходимо проверить найденные группы и исключить ситуацию, когда одинаковые
описания попадают одновременно в обучающую и тестовую выборки.

Дополнительно проверим уникальность `case_id`, наличие полных дубликатов строк
и повторяющихся значений `description`.

In [6]:
# Проверяем уникальность идентификаторов
duplicate_case_ids = df["case_id"].duplicated().sum()

# Проверяем полные дубликаты строк
duplicate_rows = df.duplicated().sum()

# Проверяем одинаковые description
duplicate_descriptions = df["description"].duplicated().sum()

# Выбираем записи, для которых задана группа дубликатов
marked_duplicates = df[
    df["duplicate_group"].notna()
].copy()

print("Дубликаты case_id:", duplicate_case_ids)
print("Полные дубликаты строк:", duplicate_rows)
print("Повторяющиеся description:", duplicate_descriptions)

print("\nСтрок с duplicate_group:", len(marked_duplicates))
print(
    "Количество групп дубликатов:",
    marked_duplicates["duplicate_group"].nunique()
)

print("\nТипы дубликатов:")
print(
    marked_duplicates["duplicate_type"]
    .value_counts(dropna=False)
)

Дубликаты case_id: 0
Полные дубликаты строк: 0
Повторяющиеся description: 0

Строк с duplicate_group: 10
Количество групп дубликатов: 10

Типы дубликатов:
duplicate_type
technical    10
Name: count, dtype: int64


## Разделение данных на обучающую и тестовую выборки

Для классификации используется только поле `description`.
Поле `bethesda` является целевой переменной.

Служебные поля `case_id`, `source_table`, `source_row`, `duplicate_group`
и `duplicate_type`, а также поле `conclusion`, в признаки модели не включаются.

Перед разделением было подтверждено отсутствие повторяющихся `description`,
поэтому одинаковые тексты не могут одновременно попасть в train и test.

Данные разделяются в отношении 80/20 со стратификацией по Bethesda.
Параметр `random_state=42` фиксируется для воспроизводимости эксперимента.

In [11]:
from sklearn.model_selection import train_test_split


# Вход модели - только цитологическое описание
X = df["description"]

# Целевая переменная - категория Bethesda
y = df["bethesda"]

# Стратифицированное разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Размер всей выборки:", len(X))
print("Размер train:", len(X_train))
print("Размер test:", len(X_test))

print("\nРаспределение классов в train:")
print(y_train.value_counts().sort_index())

print("\nРаспределение классов в test:")
print(y_test.value_counts().sort_index())

Размер всей выборки: 5613
Размер train: 4490
Размер test: 1123

Распределение классов в train:
bethesda
I       460
II     3060
III     229
IV      447
V       137
VI      157
Name: count, dtype: int64

Распределение классов в test:
bethesda
I      115
II     765
III     57
IV     112
V       35
VI      39
Name: count, dtype: int64


## Первый baseline: TF-IDF + Logistic Regression

В качестве первого воспроизводимого baseline используется комбинация
`TF-IDF + Logistic Regression`.

Параметры первого запуска намеренно оставляются простыми:

- `ngram_range=(1, 1)` - используются отдельные слова;
- `min_df=2` - исключаются термины, встретившиеся только в одном документе;
- `lowercase=True` - текст приводится к нижнему регистру;
- `class_weight=None` - балансировка классов не используется;
- `max_iter=1000` - увеличено максимальное число итераций для сходимости;
- `random_state=42` - фиксируется для воспроизводимости.

Этот запуск будет использоваться как базовая точка для сравнения последующих
вариантов модели.

In [12]:
# Параметры TF-IDF для первого baseline
tfidf_params = {
    "ngram_range": (1, 1),
    "min_df": 2,
    "lowercase": True,
}

# Параметры Logistic Regression
logreg_params = {
    "max_iter": 1000,
    "random_state": 42,
    "class_weight": None,
}

print("TF-IDF parameters:")
print(tfidf_params)

print("\nLogistic Regression parameters:")
print(logreg_params)

TF-IDF parameters:
{'ngram_range': (1, 1), 'min_df': 2, 'lowercase': True}

Logistic Regression parameters:
{'max_iter': 1000, 'random_state': 42, 'class_weight': None}
